# Частина 2: Аналіз датасету Individual Household Electric Power Consumption
Звантаження, зчитування та очищення даних (Data Cleaning).

In [1]:
import pandas as pd
import numpy as np
import timeit
import urllib.request
import zipfile
import os

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
zip_path = "household_power_consumption.zip"
txt_path = "household_power_consumption.txt"

if not os.path.exists(txt_path):
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall()

df = pd.read_csv(txt_path, sep=';', na_values=['?'], low_memory=False)

print("--- ДО ОЧИЩЕННЯ ---")
print(f"Розмір датасету: {df.shape}")
print("Кількість пропусків по колонках:\n", df.isna().sum(), "\n")

df.dropna(inplace=True)
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df.set_index('Datetime', inplace=True)
df.drop(['Date', 'Time'], axis=1, inplace=True)

print("--- ПІСЛЯ ОЧИЩЕННЯ ---")
print(f"Дані успішно очищені. Залишилось записів: {len(df)}")
display(df.head(3))

--- ДО ОЧИЩЕННЯ ---
Розмір датасету: (2075259, 9)
Кількість пропусків по колонках:
 Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64 

--- ПІСЛЯ ОЧИЩЕННЯ ---
Дані успішно очищені. Залишилось записів: 2049280


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0


### Допоміжна функція для профілювання часу (timeit)

In [2]:
def profile_task(task_func, data, task_name):
    start_time = timeit.default_timer()
    result = task_func(data)
    execution_time = timeit.default_timer() - start_time
    
    print(f"--- {task_name} ---")
    print(f"Час виконання: {execution_time:.4f} секунд")
    if isinstance(result, pd.Series):
        print("Результат (Середні значення):\n", result, "\n")
    else:
        print(f"Знайдено записів: {len(result)}\n")
    return result

### Завдання 1
**Умова:** Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.

In [3]:
def task_1_high_power(data):
    return data[data['Global_active_power'] > 5.0]

res1 = profile_task(task_1_high_power, df, "Завдання 1: Потужність > 5 кВт")
display(res1.head(3))

--- Завдання 1: Потужність > 5 кВт ---
Час виконання: 0.0072 секунд
Знайдено записів: 17547



,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,,,,
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0


### Завдання 2
**Умова:** Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер (Група 2 > Група 3).

In [4]:
def task_2_current_and_appliances(data):
    return data[(data['Global_intensity'] >= 19.0) & 
                (data['Global_intensity'] <= 20.0) & 
                (data['Sub_metering_2'] > data['Sub_metering_3'])]

res2 = profile_task(task_2_current_and_appliances, df, "Завдання 2: Струм 19-20 А + Група 2 > Групи 3")
display(res2.head(3))

--- Завдання 2: Струм 19-20 А + Група 2 > Групи 3 ---
Час виконання: 0.0127 секунд
Знайдено записів: 2509



,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,,,,
2006-12-16 18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
2006-12-17 01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
2006-12-17 01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0


### Завдання 3
**Умова:** Обрати випадковим чином 500,000 записів (без повторів), для них обчислити середні величини усіх 3-х груп споживання електричної енергії.

In [5]:
def task_3_random_sample_mean(data):
    sample = data.sample(n=500000, replace=False, random_state=42)
    return sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

res3 = profile_task(task_3_random_sample_mean, df, "Завдання 3: Середнє для 500,000 випадкових записів")

--- Завдання 3: Середнє для 500,000 випадкових записів ---
Час виконання: 0.0789 секунд
Результат (Середні значення):
 Sub_metering_1    1.119258
Sub_metering_2    1.308912
Sub_metering_3    6.452950
dtype: float64 



### Завдання 4
**Умова:** Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на групу 2 (вона є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [7]:
def task_4_complex_filter(data):

    res = data[(data.index.hour >= 18) & (data['Global_active_power'] > 6.0)]
    
    res = res[(res['Sub_metering_2'] > res['Sub_metering_1']) & 
              (res['Sub_metering_2'] > res['Sub_metering_3'])]
    
    half_idx = len(res) // 2
    first_half = res.iloc[:half_idx]
    second_half = res.iloc[half_idx:]
    
    res1 = first_half.iloc[::3]
    res2 = second_half.iloc[::4]
    
    return pd.concat([res1, res2])

res4 = profile_task(task_4_complex_filter, df, "Завдання 4: Складний вечірній фільтр")
display(res4.head(3))

--- Завдання 4: Складний вечірній фільтр ---
Час виконання: 0.0499 секунд
Знайдено записів: 310



,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,,,,
2006-12-16 18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
2006-12-16 18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
2006-12-28 20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0


### Статистичні операції та перетворення
**Умова:** Пронормувати та стандартизувати датасет, підрахувати кореляції, провести One Hot Encoding.

In [8]:
numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity']

# 1. Нормування та стандартизація
df_normalized = (df[numeric_cols] - df[numeric_cols].min()) / (df[numeric_cols].max() - df[numeric_cols].min())
df_standardized = (df[numeric_cols] - df[numeric_cols].mean()) / df[numeric_cols].std()

print("--- Стандартизовані дані (перші 3 рядки) ---")
display(df_standardized.head(3))

# 2. Кореляції Пірсона та Спірмена
pearson_corr = df['Global_active_power'].corr(df['Global_intensity'], method='pearson')
spearman_corr = df['Global_active_power'].corr(df['Global_intensity'], method='spearman')

print(f"Кореляція Пірсона (Потужність vs Струм): {pearson_corr:.4f}")
print(f"Кореляція Спірмена (Потужність vs Струм): {spearman_corr:.4f}\n")

# 3. One Hot Encoding
df_encoded = df.copy()
df_encoded['Day_of_week'] = df_encoded.index.day_name()
df_ohe = pd.get_dummies(df_encoded, columns=['Day_of_week'], prefix='Day')

print("--- Стовпці після One Hot Encoding (перші 3 рядки) ---")
display(df_ohe.filter(like='Day_').head(3))

--- Стандартизовані дані (перші 3 рядки) ---


,Global_active_power,Global_reactive_power,Voltage,Global_intensity
Datetime,,,,
2006-12-16 17:24:00,2.955076,2.610720,-1.851816,3.098788
2006-12-16 17:25:00,4.037084,2.770405,-2.225274,4.133799
2006-12-16 17:26:00,4.050325,3.320431,-2.330213,4.133799


Кореляція Пірсона (Потужність vs Струм): 0.9989
Кореляція Спірмена (Потужність vs Струм): 0.9954

--- Стовпці після One Hot Encoding (перші 3 рядки) ---


,Day_Friday,Day_Monday,Day_Saturday,Day_Sunday,Day_Thursday,Day_Tuesday,Day_Wednesday
Datetime,,,,,,,
2006-12-16 17:24:00,False,False,True,False,False,False,False
2006-12-16 17:25:00,False,False,True,False,False,False,False
2006-12-16 17:26:00,False,False,True,False,False,False,False
